# 📓 Notebook 08 — Inventory Recommendation Engine

Now RetailIQ moves from:

```text
Notebook 05 → Forecast demand
Notebook 06 → Predict stockout risk
Notebook 07 → Explain stockout risk
Notebook 08 → Recommend inventory action
```

---

## 🎯 Goal

For each **Store + Product + Date**, calculate:

* 📈 **Expected demand**
* 🛡️ **Safety stock**
* 📍 **Reorder point**
* 📦 **Current inventory**
* 🧮 **Inventory position**
* 🛒 **Recommended order quantity**
* 🚦 **Action**
* ⚠️ **Stockout risk**

---

## 🔄 Inventory Recommendation Flow

```text
                 Forecasted Demand
                        ↓
                 Safety Stock
                        ↓
                 Reorder Point
                        ↓
                Current Inventory
                        ↓
                Inventory Position
                        ↓
          ┌─────────────┴─────────────┐
          ↓                           ↓
   Stockout Risk                Inventory Level
          ↓                           ↓
          └─────────────┬─────────────┘
                        ↓
             Recommended Action
                        ↓
        Order Quantity / No Order
```

---

## 🧠 Recommendation Approach

We will use a **transparent rule-based inventory engine** rather than letting an ML model arbitrarily decide the order quantity.

This makes the recommendation:

* 🔍 **Explainable**
* 📏 **Rule-driven**
* 🧮 **Easy to validate**
* 🏪 **Business-friendly**
* 🔄 **Consistent across stores and products**

---
https://chatgpt.com/s/w_6ab7a30259f0819193bace39864c8b89

## 🎯 Final Output

For every **Store + Product + Date**, Notebook 08 should produce a structured inventory recommendation containing:

```text
Store
Product
Date
Expected Demand
Safety Stock
Reorder Point
Current Inventory
Inventory Position
Recommended Order Quantity
Action
Stockout Risk
```

### 🏁 Final Business Question

> **“Given expected demand, inventory levels, safety stock, reorder requirements, and stockout risk, what inventory action should RetailIQ take today?”**


### Step 1 — Import Libraries
Why

We need the processed forecasting and stockout outputs to build the recommendation engine.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Libraries imported successfully.")

Libraries imported successfully.


### Step 2 — Load Feature-Engineered Data
Why

This provides the current inventory and historical demand information needed for inventory calculations.

In [2]:
df = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print(
    "Date range:",
    df["Date"].min(),
    "to",
    df["Date"].max()
)

Dataset shape: (76000, 40)
Date range: 2022-01-01 00:00:00 to 2024-01-30 00:00:00


### Step 3 — Load Demand Forecasting Results
Why

The forecasting model provides the expected demand that the inventory engine will plan against.

In [3]:
forecast_results = pd.read_csv(
    "../data/processed/demand_forecasting_results.csv"
)

print(
    "Forecast results shape:",
    forecast_results.shape
)

display(
    forecast_results.head()
)

Forecast results shape: (6, 4)


,Model,MAE,RMSE,R2
0,XGBoost — Clean Features,10.343194,14.796864,0.887815
1,Random Forest,20.145341,27.553213,0.611010
2,Linear Regression,25.351812,32.695137,0.452278
3,7-Day Moving Average,31.608601,40.626437,NaN
4,XGBoost — Historical Only,32.424145,40.712853,0.150708


### Step 4 — Inspect Forecast Columns
Why

We need to identify the prediction and actual-demand columns before joining them to inventory data.

In [4]:
print(
    forecast_results.columns.tolist()
)

['Model', 'MAE', 'RMSE', 'R2']


### Step 5 — Load Stockout Predictions
Why

Stockout probability lets the recommendation engine identify higher-risk inventory situations.

In [5]:
stockout_predictions = pd.read_csv(
    "../data/processed/stockout_predictions.csv"
)

stockout_predictions["Date"] = pd.to_datetime(
    stockout_predictions["Date"]
)

print(
    "Stockout prediction shape:",
    stockout_predictions.shape
)

display(
    stockout_predictions.head()
)

Stockout prediction shape: (14700, 13)


,Date,Store ID,Product ID,Category,Inventory Level,Units Sold,Units Ordered,Demand_Lag_1,Demand_Rolling_Mean_7,Inventory_Coverage_Days,stockout_next_7_days,XGB_Stockout_Probability,XGB_Predicted_Stockout
0,2023-09-06,S005,P0007,Groceries,151,151,361,117.0,111.714286,1.351662,0,0.166917,0
1,2023-09-06,S005,P0019,Furniture,190,43,0,74.0,85.714286,2.216667,0,0.033246,0
2,2023-09-06,S001,P0012,Groceries,254,67,0,174.0,138.428571,1.834881,0,0.012453,0
3,2023-09-06,S005,P0016,Toys,326,120,553,100.0,81.571429,3.996497,0,0.105787,0
4,2023-09-06,S003,P0002,Groceries,125,59,0,96.0,108.428571,1.152833,0,0.023305,0


### Step 6 — Inspect Stockout Columns
Why

We need the stockout probability and prediction fields for the recommendation engine.

In [6]:
print(
    stockout_predictions.columns.tolist()
)

['Date', 'Store ID', 'Product ID', 'Category', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand_Lag_1', 'Demand_Rolling_Mean_7', 'Inventory_Coverage_Days', 'stockout_next_7_days', 'XGB_Stockout_Probability', 'XGB_Predicted_Stockout']


### Step 7 — Use the Final Product Risk Predictions
Why

This file contains the business-facing risk information created in Notebook 07.

In [7]:
risk_predictions = pd.read_csv(
    "../data/processed/stockout_product_explanations.csv"
)

risk_predictions["Date"] = pd.to_datetime(
    risk_predictions["Date"]
)

print(
    "Risk prediction shape:",
    risk_predictions.shape
)

display(
    risk_predictions.head()
)

Risk prediction shape: (14700, 15)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand_Lag_1,Demand_Rolling_Mean_7,Inventory_Coverage_Days,stockout_next_7_days,Stockout_Probability,Predicted_Stockout,Risk_Level
0,2023-11-25,S004,P0016,Furniture,West,170,87,234,36.0,75.571429,2.249527,0,0.846082,1,Very High
1,2024-01-12,S005,P0009,Groceries,North,136,136,764,115.0,84.285714,1.613559,0,0.839340,1,Very High
2,2023-09-23,S004,P0002,Groceries,West,254,242,488,101.0,100.285714,2.532764,0,0.807737,1,Very High
3,2023-12-25,S001,P0006,Toys,North,149,67,366,72.0,91.285714,1.632238,0,0.803426,1,Very High
4,2024-01-12,S005,P0015,Groceries,North,221,189,412,109.0,78.285714,2.822993,0,0.802900,1,Very High


### Step 8 — Inspect Current Inventory Data
Why

The recommendation must compare expected future demand against inventory currently available.

In [8]:
inventory_columns = [
    "Date",
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand"
]

inventory_data = df[
    inventory_columns
].copy()

print(
    "Inventory records:",
    len(inventory_data)
)

display(
    inventory_data.head()
)

Inventory records: 76000


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,115
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,84
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,132
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,67
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,110


### Step 9 — Create Planning Demand
Why

We need a realistic short-term demand estimate for inventory planning.

For this first version, we will use the 7-day historical rolling demand as the planning demand signal.

In [9]:
inventory_data[
    "Planning_Daily_Demand"
] = (
    df[
        "Demand_Rolling_Mean_7"
    ]
    .loc[inventory_data.index]
)

inventory_data[
    "Planning_Daily_Demand"
] = inventory_data[
    "Planning_Daily_Demand"
].fillna(
    inventory_data["Demand"]
)

print(
    inventory_data[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Inventory Level",
            "Planning_Daily_Demand"
        ]
    ].head(10)
)

        Date Store ID Product ID  Inventory Level  Planning_Daily_Demand
0 2022-01-01     S001      P0001              195             115.000000
1 2022-01-02     S001      P0001               93              84.000000
2 2022-01-03     S001      P0001              274             132.000000
3 2022-01-04     S001      P0001              132              67.000000
4 2022-01-05     S001      P0001              319             110.000000
5 2022-01-06     S001      P0001              190             146.000000
6 2022-01-07     S001      P0001               92              87.000000
7 2022-01-08     S001      P0001              308             105.857143
8 2022-01-09     S001      P0001              212             105.571429
9 2022-01-10     S001      P0001              403             106.000000


### Step 10 — Define Planning Horizon
Why

RetailIQ needs to decide how much stock is required for the upcoming replenishment period.

We will use a 7-day planning horizon.

In [10]:
PLANNING_HORIZON_DAYS = 7

print(
    "Planning horizon:",
    PLANNING_HORIZON_DAYS,
    "days"
)

Planning horizon: 7 days


### Step 11 — Calculate Expected 7-Day Demand
Why

This estimates how much inventory the product may require over the next week.

In [11]:
inventory_data[
    "Expected_7_Day_Demand"
] = (
    inventory_data[
        "Planning_Daily_Demand"
    ]
    * PLANNING_HORIZON_DAYS
)

display(
    inventory_data[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Planning_Daily_Demand",
            "Expected_7_Day_Demand"
        ]
    ].head(10)
)

,Date,Store ID,Product ID,Planning_Daily_Demand,Expected_7_Day_Demand
0,2022-01-01,S001,P0001,115.000000,805.0
1,2022-01-02,S001,P0001,84.000000,588.0
2,2022-01-03,S001,P0001,132.000000,924.0
3,2022-01-04,S001,P0001,67.000000,469.0
4,2022-01-05,S001,P0001,110.000000,770.0
5,2022-01-06,S001,P0001,146.000000,1022.0
6,2022-01-07,S001,P0001,87.000000,609.0
7,2022-01-08,S001,P0001,105.857143,741.0
8,2022-01-09,S001,P0001,105.571429,739.0
9,2022-01-10,S001,P0001,106.000000,742.0


### Step 12 — Calculate Demand Variability
Why

Stable demand needs less protection stock than highly variable demand.

In [12]:
inventory_data[
    "Demand_Std_7_Day"
] = (
    df[
        "Demand_Rolling_Std_7"
    ]
    .loc[inventory_data.index]
)

inventory_data[
    "Demand_Std_7_Day"
] = inventory_data[
    "Demand_Std_7_Day"
].fillna(0)

display(
    inventory_data[
        [
            "Product ID",
            "Demand_Std_7_Day"
        ]
    ].head(10)
)

,Product ID,Demand_Std_7_Day
0,P0001,0.000000
1,P0001,0.000000
2,P0001,0.000000
3,P0001,0.000000
4,P0001,0.000000
5,P0001,0.000000
6,P0001,0.000000
7,P0001,28.103550
8,P0001,28.005102
9,P0001,27.640550


### Step 13 — Define Service Level
Why

Safety stock should provide protection against demand uncertainty.

We will initially use a 95% service level, corresponding approximately to a Z-score of 1.65.

In [13]:
SERVICE_LEVEL = 0.95
Z_SCORE = 1.65

print(
    "Service level:",
    SERVICE_LEVEL
)

print(
    "Z-score:",
    Z_SCORE
)

Service level: 0.95
Z-score: 1.65


### Step 14 — Calculate Safety Stock
Why

Safety stock protects against unexpected demand fluctuations.

In [14]:
inventory_data[
    "Safety_Stock"
] = (
    Z_SCORE
    * inventory_data[
        "Demand_Std_7_Day"
    ]
    * np.sqrt(
        PLANNING_HORIZON_DAYS
    )
)

inventory_data[
    "Safety_Stock"
] = np.ceil(
    inventory_data[
        "Safety_Stock"
]
)

display(
    inventory_data[
        [
            "Product ID",
            "Demand_Std_7_Day",
            "Safety_Stock"
        ]
    ].head(10)
)

,Product ID,Demand_Std_7_Day,Safety_Stock
0,P0001,0.000000,0.0
1,P0001,0.000000,0.0
2,P0001,0.000000,0.0
3,P0001,0.000000,0.0
4,P0001,0.000000,0.0
5,P0001,0.000000,0.0
6,P0001,0.000000,0.0
7,P0001,28.103550,123.0
8,P0001,28.005102,123.0
9,P0001,27.640550,121.0


### Step 15 — Calculate Reorder Point
Why

The reorder point indicates the inventory level at which replenishment should be triggered.

In [15]:
inventory_data[
    "Reorder_Point"
] = (
    inventory_data[
        "Expected_7_Day_Demand"
    ]
    + inventory_data[
        "Safety_Stock"
    ]
)

inventory_data[
    "Reorder_Point"
] = np.ceil(
    inventory_data[
        "Reorder_Point"
    ]
)

display(
    inventory_data[
        [
            "Product ID",
            "Expected_7_Day_Demand",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ].head(10)
)

,Product ID,Expected_7_Day_Demand,Safety_Stock,Reorder_Point
0,P0001,805.0,0.0,805.0
1,P0001,588.0,0.0,588.0
2,P0001,924.0,0.0,924.0
3,P0001,469.0,0.0,469.0
4,P0001,770.0,0.0,770.0
5,P0001,1022.0,0.0,1022.0
6,P0001,609.0,0.0,609.0
7,P0001,741.0,123.0,864.0
8,P0001,739.0,123.0,862.0
9,P0001,742.0,121.0,863.0


### Step 16 — Calculate Inventory Position
Why

Inventory position considers stock currently available plus stock already ordered.

In [16]:
inventory_data[
    "Inventory_Position"
] = (
    inventory_data[
        "Inventory Level"
    ]
    + inventory_data[
        "Units Ordered"
    ]
)

display(
    inventory_data[
        [
            "Product ID",
            "Inventory Level",
            "Units Ordered",
            "Inventory_Position"
        ]
    ].head(10)
)

,Product ID,Inventory Level,Units Ordered,Inventory_Position
0,P0001,195,252,447
1,P0001,93,0,93
2,P0001,274,229,503
3,P0001,132,0,132
4,P0001,319,0,319
5,P0001,190,308,498
6,P0001,92,0,92
7,P0001,308,248,556
8,P0001,212,0,212
9,P0001,403,0,403


### Step 17 — Calculate Raw Recommended Order
Why

We order only enough inventory to bring the inventory position up to the reorder requirement.

In [17]:
inventory_data[
    "Raw_Recommended_Order"
] = (
    inventory_data[
        "Reorder_Point"
    ]
    - inventory_data[
        "Inventory_Position"
    ]
)

inventory_data[
    "Raw_Recommended_Order"
] = (
    inventory_data[
        "Raw_Recommended_Order"
    ].clip(lower=0)
)

display(
    inventory_data[
        [
            "Product ID",
            "Inventory_Position",
            "Reorder_Point",
            "Raw_Recommended_Order"
        ]
    ].head(10)
)

,Product ID,Inventory_Position,Reorder_Point,Raw_Recommended_Order
0,P0001,447,805.0,358.0
1,P0001,93,588.0,495.0
2,P0001,503,924.0,421.0
3,P0001,132,469.0,337.0
4,P0001,319,770.0,451.0
5,P0001,498,1022.0,524.0
6,P0001,92,609.0,517.0
7,P0001,556,864.0,308.0
8,P0001,212,862.0,650.0
9,P0001,403,863.0,460.0


### Step 18 — Round Recommended Quantity
Why

Inventory orders must be whole units.

In [18]:
inventory_data[
    "Recommended_Order_Qty"
] = np.ceil(
    inventory_data[
        "Raw_Recommended_Order"
    ]
).astype(int)

display(
    inventory_data[
        [
            "Product ID",
            "Recommended_Order_Qty"
        ]
    ].head(10)
)

,Product ID,Recommended_Order_Qty
0,P0001,358
1,P0001,495
2,P0001,421
3,P0001,337
4,P0001,451
5,P0001,524
6,P0001,517
7,P0001,308
8,P0001,650
9,P0001,460


### Step 19 — Calculate Inventory Coverage
Why

Coverage tells the manager approximately how many days the current inventory can support.

In [19]:
inventory_data[
    "Inventory_Coverage_Days"
] = np.where(
    inventory_data[
        "Planning_Daily_Demand"
    ] > 0,
    inventory_data[
        "Inventory_Position"
    ]
    / inventory_data[
        "Planning_Daily_Demand"
    ],
    np.inf
)

display(
    inventory_data[
        [
            "Product ID",
            "Inventory_Position",
            "Planning_Daily_Demand",
            "Inventory_Coverage_Days"
        ]
    ].head(10)
)

,Product ID,Inventory_Position,Planning_Daily_Demand,Inventory_Coverage_Days
0,P0001,447,115.000000,3.886957
1,P0001,93,84.000000,1.107143
2,P0001,503,132.000000,3.810606
3,P0001,132,67.000000,1.970149
4,P0001,319,110.000000,2.900000
5,P0001,498,146.000000,3.410959
6,P0001,92,87.000000,1.057471
7,P0001,556,105.857143,5.252362
8,P0001,212,105.571429,2.008119
9,P0001,403,106.000000,3.801887


### Step 20 — Create Base Inventory Action
Why

RetailIQ needs a simple operational decision that a manager can understand immediately.

In [20]:
inventory_data[
    "Inventory_Action"
] = np.select(
    [
        inventory_data[
            "Inventory_Position"
        ]
        <= inventory_data[
            "Reorder_Point"
        ],

        inventory_data[
            "Inventory_Coverage_Days"
        ] > 30
    ],
    [
        "REORDER",
        "REVIEW OVERSTOCK"
    ],
    default="MONITOR"
)

display(
    inventory_data[
        "Inventory_Action"
    ].value_counts()
)

Inventory_Action
REORDER             70731
MONITOR              5254
REVIEW OVERSTOCK       15
Name: count, dtype: int64

### Step 21 — Create Recommendation Dataset
Why

This becomes the central table used by the future Power BI dashboard and API.

In [21]:
recommendation_columns = [
    "Date",
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Planning_Daily_Demand",
    "Expected_7_Day_Demand",
    "Demand_Std_7_Day",
    "Safety_Stock",
    "Reorder_Point",
    "Inventory_Position",
    "Inventory_Coverage_Days",
    "Recommended_Order_Qty",
    "Inventory_Action"
]

recommendations = inventory_data[
    recommendation_columns
].copy()

print(
    "Recommendation dataset shape:",
    recommendations.shape
)

display(
    recommendations.head(10)
)

Recommendation dataset shape: (76000, 17)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Planning_Daily_Demand,Expected_7_Day_Demand,Demand_Std_7_Day,Safety_Stock,Reorder_Point,Inventory_Position,Inventory_Coverage_Days,Recommended_Order_Qty,Inventory_Action
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,115.000000,805.0,0.000000,0.0,805.0,447,3.886957,358,REORDER
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,84.000000,588.0,0.000000,0.0,588.0,93,1.107143,495,REORDER
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,132.000000,924.0,0.000000,0.0,924.0,503,3.810606,421,REORDER
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,67.000000,469.0,0.000000,0.0,469.0,132,1.970149,337,REORDER
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,110.000000,770.0,0.000000,0.0,770.0,319,2.900000,451,REORDER
5,2022-01-06,S001,P0001,Electronics,North,190,98,308,146.000000,1022.0,0.000000,0.0,1022.0,498,3.410959,524,REORDER
6,2022-01-07,S001,P0001,Electronics,North,92,92,0,87.000000,609.0,0.000000,0.0,609.0,92,1.057471,517,REORDER
7,2022-01-08,S001,P0001,Electronics,North,308,96,248,105.857143,741.0,28.103550,123.0,864.0,556,5.252362,308,REORDER
8,2022-01-09,S001,P0001,Electronics,North,212,57,0,105.571429,739.0,28.005102,123.0,862.0,212,2.008119,650,REORDER
9,2022-01-10,S001,P0001,Electronics,North,403,88,0,106.000000,742.0,27.640550,121.0,863.0,403,3.801887,460,REORDER


### Step 22 — Add Stockout Risk
Why

The final recommendation should combine inventory need with predicted stockout risk.

In [22]:
risk_columns = [
    "Date",
    "Store ID",
    "Product ID",
    "Stockout_Probability",
    "Predicted_Stockout"
]

risk_data = risk_predictions[
    risk_columns
].copy()

print(
    risk_data.head()
)

        Date Store ID Product ID  Stockout_Probability  Predicted_Stockout
0 2023-11-25     S004      P0016              0.846082                   1
1 2024-01-12     S005      P0009              0.839340                   1
2 2023-09-23     S004      P0002              0.807737                   1
3 2023-12-25     S001      P0006              0.803426                   1
4 2024-01-12     S005      P0015              0.802900                   1


### Step 23 — Merge Stockout Risk
Why

This connects the predictive risk model with the inventory recommendation engine.

In [23]:
recommendations = recommendations.merge(
    risk_data,
    on=[
        "Date",
        "Store ID",
        "Product ID"
    ],
    how="left"
)

print(
    "After risk merge:",
    recommendations.shape
)

print(
    "Missing stockout probabilities:",
    recommendations[
        "Stockout_Probability"
    ].isna().sum()
)

After risk merge: (76000, 19)
Missing stockout probabilities: 61300


### Step 24 — Fill Missing Risk Values
Why

Some records may not have a corresponding stockout prediction, so we need a safe default.

In [24]:
recommendations[
    "Stockout_Probability"
] = recommendations[
    "Stockout_Probability"
].fillna(0)

recommendations[
    "Predicted_Stockout"
] = recommendations[
    "Predicted_Stockout"
].fillna(0).astype(int)

### Step 25 — Create Risk-Aware Action
Why

A high predicted stockout probability should receive more urgent attention.

In [25]:
recommendations[
    "Risk_Aware_Action"
] = np.select(
    [
        recommendations[
            "Stockout_Probability"
        ] >= 0.75,

        recommendations[
            "Stockout_Probability"
        ] >= 0.50,

        recommendations[
            "Inventory_Action"
        ] == "REORDER",

        recommendations[
            "Inventory_Action"
        ] == "REVIEW OVERSTOCK"
    ],
    [
        "URGENT REORDER",
        "PRIORITY REORDER",
        "REORDER",
        "REVIEW OVERSTOCK"
    ],
    default="MONITOR"
)

display(
    recommendations[
        "Risk_Aware_Action"
    ].value_counts()
)

Risk_Aware_Action
REORDER             70633
MONITOR              5243
PRIORITY REORDER       99
REVIEW OVERSTOCK       15
URGENT REORDER         10
Name: count, dtype: int64

### Step 26 — Create Risk Level
Why

Risk levels make the recommendation easier to consume in Power BI and the future GenAI Copilot.

In [26]:
recommendations[
    "Risk_Level"
] = pd.cut(
    recommendations[
        "Stockout_Probability"
    ],
    bins=[
        -np.inf,
        0.20,
        0.50,
        0.75,
        np.inf
    ],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

display(
    recommendations[
        "Risk_Level"
    ].value_counts()
)

Risk_Level
Low          75220
Moderate       671
High            99
Very High       10
Name: count, dtype: int64

### Step 27 — Calculate Potential Stockout Gap
Why

This estimates how far inventory position is below the expected requirement.

In [27]:
recommendations[
    "Inventory_Gap_to_Reorder"
] = (
    recommendations[
        "Reorder_Point"
    ]
    - recommendations[
        "Inventory_Position"
    ]
).clip(lower=0)

display(
    recommendations[
        [
            "Product ID",
            "Inventory_Position",
            "Reorder_Point",
            "Inventory_Gap_to_Reorder"
        ]
    ].head(10)
)

,Product ID,Inventory_Position,Reorder_Point,Inventory_Gap_to_Reorder
0,P0001,447,805.0,358.0
1,P0001,93,588.0,495.0
2,P0001,503,924.0,421.0
3,P0001,132,469.0,337.0
4,P0001,319,770.0,451.0
5,P0001,498,1022.0,524.0
6,P0001,92,609.0,517.0
7,P0001,556,864.0,308.0
8,P0001,212,862.0,650.0
9,P0001,403,863.0,460.0


### Step 28 — Create Final Recommendation Score
Why

This creates a transparent urgency indicator combining stockout probability and replenishment need.

In [28]:
recommendations[
    "Recommendation_Urgency"
] = (
    recommendations[
        "Stockout_Probability"
    ]
    * 100
    + np.where(
        recommendations[
            "Recommended_Order_Qty"
        ] > 0,
        20,
        0
    )
)

recommendations[
    "Recommendation_Urgency"
] = recommendations[
    "Recommendation_Urgency"
].round(2)

display(
    recommendations[
        [
            "Product ID",
            "Stockout_Probability",
            "Recommended_Order_Qty",
            "Recommendation_Urgency"
        ]
    ].head(10)
)

,Product ID,Stockout_Probability,Recommended_Order_Qty,Recommendation_Urgency
0,P0001,0.0,358,20.0
1,P0001,0.0,495,20.0
2,P0001,0.0,421,20.0
3,P0001,0.0,337,20.0
4,P0001,0.0,451,20.0
5,P0001,0.0,524,20.0
6,P0001,0.0,517,20.0
7,P0001,0.0,308,20.0
8,P0001,0.0,650,20.0
9,P0001,0.0,460,20.0


### Step 29 — Sort Highest Priority Recommendations
Why

Managers need the most urgent inventory issues first.

In [29]:
priority_recommendations = (
    recommendations
    .sort_values(
        "Recommendation_Urgency",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    priority_recommendations.head(20)
)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Planning_Daily_Demand,Expected_7_Day_Demand,Demand_Std_7_Day,Safety_Stock,Reorder_Point,Inventory_Position,Inventory_Coverage_Days,Recommended_Order_Qty,Inventory_Action,Stockout_Probability,Predicted_Stockout,Risk_Aware_Action,Risk_Level,Inventory_Gap_to_Reorder,Recommendation_Urgency
0,2023-11-25,S004,P0016,Furniture,West,170,87,234,75.571429,529.0,20.321230,89.0,618.0,404,5.345936,214,REORDER,0.846082,1,URGENT REORDER,Very High,214.0,104.61
1,2023-09-23,S004,P0002,Groceries,West,254,242,488,100.285714,702.0,46.241087,202.0,904.0,742,7.398860,162,REORDER,0.807737,1,URGENT REORDER,Very High,162.0,100.77
2,2023-12-25,S001,P0006,Toys,North,149,67,366,91.285714,639.0,28.697934,126.0,765.0,515,5.641628,250,REORDER,0.803426,1,URGENT REORDER,Very High,250.0,100.34
3,2024-01-12,S005,P0015,Groceries,North,221,189,412,78.285714,548.0,26.017393,114.0,662.0,633,8.085766,29,REORDER,0.802900,1,URGENT REORDER,Very High,29.0,100.29
4,2023-09-19,S005,P0016,Toys,North,584,130,0,114.142857,799.0,28.858439,126.0,925.0,584,5.116395,341,REORDER,0.788939,1,URGENT REORDER,Very High,341.0,98.89
5,2023-09-28,S004,P0016,Furniture,West,110,103,292,77.285714,541.0,16.049032,71.0,612.0,402,5.201479,210,REORDER,0.783365,1,URGENT REORDER,Very High,210.0,98.34
6,2023-09-22,S005,P0016,Toys,North,264,94,327,104.285714,730.0,22.786797,100.0,830.0,591,5.667123,239,REORDER,0.769643,1,URGENT REORDER,Very High,239.0,96.96
7,2023-10-24,S005,P0009,Groceries,North,446,38,0,99.142857,694.0,23.575613,103.0,797.0,446,4.498559,351,REORDER,0.756334,1,URGENT REORDER,Very High,351.0,95.63
8,2023-09-17,S001,P0016,Groceries,North,475,60,0,124.714286,873.0,36.768258,161.0,1034.0,475,3.808706,559,REORDER,0.715860,1,PRIORITY REORDER,High,559.0,91.59
9,2024-01-30,S005,P0009,Groceries,North,136,93,0,90.000000,630.0,29.793735,131.0,761.0,136,1.511111,625,REORDER,0.691300,1,PRIORITY REORDER,High,625.0,89.13


### Step 30 — Analyze Recommended Order Quantity
Why

This checks how much inventory RetailIQ recommends replenishing.

In [30]:
print(
    "Total recommended units:",
    recommendations[
        "Recommended_Order_Qty"
    ].sum()
)

print(
    "Average recommended order:",
    recommendations[
        "Recommended_Order_Qty"
    ].mean()
)

print(
    "Maximum recommended order:",
    recommendations[
        "Recommended_Order_Qty"
    ].max()
)

Total recommended units: 38596639
Average recommended order: 507.85051315789474
Maximum recommended order: 1726


### Step 31 — Analyze Actions

In [31]:
action_summary = (
    recommendations[
        "Risk_Aware_Action"
    ]
    .value_counts()
    .to_frame(
        "Record_Count"
    )
)

action_summary[
    "Percentage"
] = (
    action_summary[
        "Record_Count"
    ]
    / len(recommendations)
    * 100
)

action_summary[
    "Percentage"
] = action_summary[
    "Percentage"
].round(2)

display(
    action_summary
)

,Record_Count,Percentage
Risk_Aware_Action,,
REORDER,70633,92.94
MONITOR,5243,6.90
PRIORITY REORDER,99,0.13
REVIEW OVERSTOCK,15,0.02
URGENT REORDER,10,0.01


### Step 32 — Analyze Recommendations by Category
Why

This shows where inventory pressure is concentrated across the business.

In [32]:
category_summary = (
    recommendations
    .groupby("Category")
    .agg(
        Records=("Product ID", "count"),
        Recommended_Units=(
            "Recommended_Order_Qty",
            "sum"
        ),
        Avg_Risk=(
            "Stockout_Probability",
            "mean"
        ),
        Avg_Coverage=(
            "Inventory_Coverage_Days",
            "mean"
        )
    )
    .sort_values(
        "Recommended_Units",
        ascending=False
    )
)

display(
    category_summary
)

,Records,Recommended_Units,Avg_Risk,Avg_Coverage
Category,,,,
Groceries,30400,17488859,0.010325,3.928973
Clothing,12160,7114344,0.007526,3.421834
Toys,10640,5257729,0.006457,3.766693
Furniture,13680,4447771,0.007017,4.530615
Electronics,9120,4287936,0.006574,4.009998


### Step 33 — Analyze Recommendations by Store
Why

This identifies stores requiring the most replenishment attention.

In [34]:
store_summary = (
    recommendations
    .groupby("Store ID")
    .agg(
        Records=("Product ID", "count"),
        Recommended_Units=(
            "Recommended_Order_Qty",
            "sum"
        ),
        Avg_Risk=(
            "Stockout_Probability",
            "mean"
        ),
        Avg_Coverage=(
            "Inventory_Coverage_Days",
            "mean"
        )
    )
    .sort_values(
        "Recommended_Units",
        ascending=False
    )
)

display(
    store_summary
)

,Records,Recommended_Units,Avg_Risk,Avg_Coverage
Store ID,,,,
S005,15200,8151072,0.010294,3.746457
S001,15200,7794739,0.006172,3.757056
S003,15200,7714037,0.007838,4.130345
S002,15200,7555723,0.009067,4.231860
S004,15200,7381068,0.008080,3.849931


### Step 34 — Identify Critical Recommendations
Why

This gives us a manager-ready list of products requiring immediate attention.

In [35]:
critical_recommendations = recommendations[
    (
        recommendations[
            "Stockout_Probability"
        ] >= 0.50
    )
    |
    (
        recommendations[
            "Risk_Aware_Action"
        ] == "URGENT REORDER"
    )
].copy()

critical_recommendations = (
    critical_recommendations
    .sort_values(
        "Stockout_Probability",
        ascending=False
    )
)

print(
    "Critical records:",
    len(critical_recommendations)
)

display(
    critical_recommendations.head(20)
)

Critical records: 109


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Planning_Daily_Demand,Expected_7_Day_Demand,Demand_Std_7_Day,Safety_Stock,Reorder_Point,Inventory_Position,Inventory_Coverage_Days,Recommended_Order_Qty,Inventory_Action,Stockout_Probability,Predicted_Stockout,Risk_Aware_Action,Risk_Level,Inventory_Gap_to_Reorder,Recommendation_Urgency
57693,2023-11-25,S004,P0016,Furniture,West,170,87,234,75.571429,529.0,20.321230,89.0,618.0,404,5.345936,214,REORDER,0.846082,1,URGENT REORDER,Very High,214.0,104.61
67621,2024-01-12,S005,P0009,Groceries,North,136,136,764,84.285714,590.0,21.891507,96.0,686.0,900,10.677966,0,MONITOR,0.839340,1,URGENT REORDER,Very High,0.0,83.93
46990,2023-09-23,S004,P0002,Groceries,West,254,242,488,100.285714,702.0,46.241087,202.0,904.0,742,7.398860,162,REORDER,0.807737,1,URGENT REORDER,Very High,162.0,100.77
4523,2023-12-25,S001,P0006,Toys,North,149,67,366,91.285714,639.0,28.697934,126.0,765.0,515,5.641628,250,REORDER,0.803426,1,URGENT REORDER,Very High,250.0,100.34
72181,2024-01-12,S005,P0015,Groceries,North,221,189,412,78.285714,548.0,26.017393,114.0,662.0,633,8.085766,29,REORDER,0.802900,1,URGENT REORDER,Very High,29.0,100.29
72826,2023-09-19,S005,P0016,Toys,North,584,130,0,114.142857,799.0,28.858439,126.0,925.0,584,5.116395,341,REORDER,0.788939,1,URGENT REORDER,Very High,341.0,98.89
57635,2023-09-28,S004,P0016,Furniture,West,110,103,292,77.285714,541.0,16.049032,71.0,612.0,402,5.201479,210,REORDER,0.783365,1,URGENT REORDER,Very High,210.0,98.34
41692,2023-10-15,S003,P0015,Groceries,East,325,154,722,104.000000,728.0,20.696215,91.0,819.0,1047,10.067308,0,MONITOR,0.773043,1,URGENT REORDER,Very High,0.0,77.30
72829,2023-09-22,S005,P0016,Toys,North,264,94,327,104.285714,730.0,22.786797,100.0,830.0,591,5.667123,239,REORDER,0.769643,1,URGENT REORDER,Very High,239.0,96.96
67541,2023-10-24,S005,P0009,Groceries,North,446,38,0,99.142857,694.0,23.575613,103.0,797.0,446,4.498559,351,REORDER,0.756334,1,URGENT REORDER,Very High,351.0,95.63


### Step 35 — Create Manager-Friendly Explanation
Why

The final system should explain the recommendation in plain business language.

In [36]:
def create_recommendation_reason(row):

    reasons = []

    if row["Stockout_Probability"] >= 0.75:
        reasons.append(
            "Very high predicted stockout risk"
        )

    elif row["Stockout_Probability"] >= 0.50:
        reasons.append(
            "High predicted stockout risk"
        )

    if (
        row["Inventory_Position"]
        <= row["Reorder_Point"]
    ):
        reasons.append(
            "inventory position is below reorder point"
        )

    if (
        row["Inventory_Coverage_Days"] < 7
    ):
        reasons.append(
            "less than 7 days of inventory coverage"
        )

    if (
        row["Recommended_Order_Qty"] > 0
    ):
        reasons.append(
            "additional inventory is recommended"
        )

    if not reasons:
        reasons.append(
            "inventory level currently appears sufficient"
        )

    return "; ".join(reasons)


recommendations[
    "Recommendation_Reason"
] = recommendations.apply(
    create_recommendation_reason,
    axis=1
)

display(
    recommendations[
        [
            "Product ID",
            "Stockout_Probability",
            "Recommended_Order_Qty",
            "Risk_Aware_Action",
            "Recommendation_Reason"
        ]
    ].head(20)
)

,Product ID,Stockout_Probability,Recommended_Order_Qty,Risk_Aware_Action,Recommendation_Reason
0,P0001,0.0,358,REORDER,inventory position is below reorder point; les...
1,P0001,0.0,495,REORDER,inventory position is below reorder point; les...
2,P0001,0.0,421,REORDER,inventory position is below reorder point; les...
3,P0001,0.0,337,REORDER,inventory position is below reorder point; les...
4,P0001,0.0,451,REORDER,inventory position is below reorder point; les...
5,P0001,0.0,524,REORDER,inventory position is below reorder point; les...
6,P0001,0.0,517,REORDER,inventory position is below reorder point; les...
7,P0001,0.0,308,REORDER,inventory position is below reorder point; les...
8,P0001,0.0,650,REORDER,inventory position is below reorder point; les...
9,P0001,0.0,460,REORDER,inventory position is below reorder point; les...


### Step 36 — Validate Recommendation Logic
Why

We need to make sure the engine never produces invalid negative order quantities.

In [37]:
print(
    "Negative order quantities:",
    (
        recommendations[
            "Recommended_Order_Qty"
        ] < 0
    ).sum()
)

print(
    "Null order quantities:",
    recommendations[
        "Recommended_Order_Qty"
    ].isna().sum()
)

print(
    "Null risk values:",
    recommendations[
        "Stockout_Probability"
    ].isna().sum()
)

print(
    "Null actions:",
    recommendations[
        "Risk_Aware_Action"
    ].isna().sum()
)

Negative order quantities: 0
Null order quantities: 0
Null risk values: 0
Null actions: 0


### Step 37 — Final Recommendation Table
Why

This is the main output that future Power BI, FastAPI, and GenAI components will consume.

In [38]:
final_recommendations = recommendations[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Category",
        "Region",
        "Inventory Level",
        "Units Ordered",
        "Planning_Daily_Demand",
        "Expected_7_Day_Demand",
        "Safety_Stock",
        "Reorder_Point",
        "Inventory_Position",
        "Inventory_Coverage_Days",
        "Recommended_Order_Qty",
        "Stockout_Probability",
        "Predicted_Stockout",
        "Risk_Level",
        "Inventory_Action",
        "Risk_Aware_Action",
        "Recommendation_Urgency",
        "Recommendation_Reason"
    ]
].copy()

display(
    final_recommendations.head(20)
)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Ordered,Planning_Daily_Demand,Expected_7_Day_Demand,Safety_Stock,Reorder_Point,Inventory_Position,Inventory_Coverage_Days,Recommended_Order_Qty,Stockout_Probability,Predicted_Stockout,Risk_Level,Inventory_Action,Risk_Aware_Action,Recommendation_Urgency,Recommendation_Reason
0,2022-01-01,S001,P0001,Electronics,North,195,252,115.000000,805.0,0.0,805.0,447,3.886957,358,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
1,2022-01-02,S001,P0001,Electronics,North,93,0,84.000000,588.0,0.0,588.0,93,1.107143,495,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
2,2022-01-03,S001,P0001,Electronics,North,274,229,132.000000,924.0,0.0,924.0,503,3.810606,421,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
3,2022-01-04,S001,P0001,Electronics,North,132,0,67.000000,469.0,0.0,469.0,132,1.970149,337,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
4,2022-01-05,S001,P0001,Electronics,North,319,0,110.000000,770.0,0.0,770.0,319,2.900000,451,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
5,2022-01-06,S001,P0001,Electronics,North,190,308,146.000000,1022.0,0.0,1022.0,498,3.410959,524,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
6,2022-01-07,S001,P0001,Electronics,North,92,0,87.000000,609.0,0.0,609.0,92,1.057471,517,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
7,2022-01-08,S001,P0001,Electronics,North,308,248,105.857143,741.0,123.0,864.0,556,5.252362,308,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
8,2022-01-09,S001,P0001,Electronics,North,212,0,105.571429,739.0,123.0,862.0,212,2.008119,650,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...
9,2022-01-10,S001,P0001,Electronics,North,403,0,106.000000,742.0,121.0,863.0,403,3.801887,460,0.0,0,Low,REORDER,REORDER,20.0,inventory position is below reorder point; les...


### Step 38 — Save Final Recommendations
Why

This becomes the main inventory recommendation dataset for the rest of RetailIQ.

In [39]:
final_recommendations.to_csv(
    "../data/processed/inventory_recommendations.csv",
    index=False
)

print(
    "Saved:",
    "../data/processed/inventory_recommendations.csv"
)

Saved: ../data/processed/inventory_recommendations.csv


### Step 39 — Save Category Summary

In [40]:
category_summary.to_csv(
    "../data/processed/inventory_category_summary.csv"
)

print(
    "Saved:",
    "../data/processed/inventory_category_summary.csv"
)

Saved: ../data/processed/inventory_category_summary.csv


### Step 40 — Save Store Summary

In [41]:
store_summary.to_csv(
    "../data/processed/inventory_store_summary.csv"
)

print(
    "Saved:",
    "../data/processed/inventory_store_summary.csv"
)

Saved: ../data/processed/inventory_store_summary.csv


### Step 41 — Save Critical Recommendations

In [42]:
critical_recommendations.to_csv(
    "../data/processed/critical_inventory_recommendations.csv",
    index=False
)

print(
    "Saved:",
    "../data/processed/critical_inventory_recommendations.csv"
)

Saved: ../data/processed/critical_inventory_recommendations.csv


### Step 42 — Final Notebook Summary

In [43]:
print("=" * 75)
print("RETAILIQ — NOTEBOOK 08 INVENTORY RECOMMENDATION ENGINE")
print("=" * 75)

print("\nTotal records:", len(final_recommendations))

print(
    "Total recommended units:",
    final_recommendations[
        "Recommended_Order_Qty"
    ].sum()
)

print(
    "Average recommended order:",
    round(
        final_recommendations[
            "Recommended_Order_Qty"
        ].mean(),
        2
    )
)

print(
    "Average stockout probability:",
    round(
        final_recommendations[
            "Stockout_Probability"
        ].mean(),
        4
    )
)

print("\nRisk distribution:")
print(
    final_recommendations[
        "Risk_Level"
    ].value_counts()
)

print("\nActions:")
print(
    final_recommendations[
        "Risk_Aware_Action"
    ].value_counts()
)

print("\nCritical records:")
print(
    len(critical_recommendations)
)

print("\nSaved files:")
print(
    "- inventory_recommendations.csv"
)

print(
    "- inventory_category_summary.csv"
)

print(
    "- inventory_store_summary.csv"
)

print(
    "- critical_inventory_recommendations.csv"
)

print("\nNotebook 08 completed successfully.")

RETAILIQ — NOTEBOOK 08 INVENTORY RECOMMENDATION ENGINE

Total records: 76000
Total recommended units: 38596639
Average recommended order: 507.85
Average stockout probability: 0.0083

Risk distribution:
Risk_Level
Low          75220
Moderate       671
High            99
Very High       10
Name: count, dtype: int64

Actions:
Risk_Aware_Action
REORDER             70633
MONITOR              5243
PRIORITY REORDER       99
REVIEW OVERSTOCK       15
URGENT REORDER         10
Name: count, dtype: int64

Critical records:
109

Saved files:
- inventory_recommendations.csv
- inventory_category_summary.csv
- inventory_store_summary.csv
- critical_inventory_recommendations.csv

Notebook 08 completed successfully.
